In [3]:
import json
import os
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from openai import OpenAI
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


def _normalize_text(value: object) -> str:
    """Normalize text for comparison."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _extract_phone_candidates(raw: str) -> list[str]:
    if not raw:
        return []
    cleaned = raw.replace(";", ",").replace("|", ",")
    pattern = r"\+?\d[\d\-\s\(\)\./]{6,}\d"
    candidates = re.findall(pattern, cleaned)
    if not candidates:
        return [cleaned]
    return candidates


def _normalize_phone_set(value: object, default_region: str = "CA") -> set[str]:
    """Normalize phone numbers into a set to handle multiple values in one field."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return set()
    raw = str(value).strip()
    if not raw:
        return set()

    numbers: set[str] = set()
    candidates = _extract_phone_candidates(raw)

    for candidate in candidates:
        candidate = re.split(r"(?:ext|x)\s*\d+", candidate, flags=re.IGNORECASE)[0]
        digits_only = re.sub(r"\D", "", candidate)
        if len(digits_only) < 7:
            continue
        try:
            import phonenumbers

            parsed = phonenumbers.parse(candidate, default_region)
            if phonenumbers.is_valid_number(parsed):
                numbers.add(phonenumbers.format_number(parsed, phonenumbers.PhoneNumberFormat.E164))
            else:
                numbers.add(digits_only)
        except Exception:
            numbers.add(digits_only)

    return numbers


def _column_score(a: str, b: str) -> Optional[float]:
    """Calculate similarity score between two strings."""
    if not a or not b:
        return None
    jw = JaroWinkler.similarity(a, b)
    ts = fuzz.token_set_ratio(a, b) / 100.0
    return max(jw, ts)


def _llm_judge_match(
    pairs: list[dict],
    model: Optional[str] = None,
    batch_size: int = 10,
) -> list[bool]:
    """Use LLM to judge if pairs match."""
    client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL", "gpt-5-nano")
    decisions: list[bool] = []

    system_prompt = (
        "You are a matching judge. Decide if two records describe the same physical facility. "
        "Return JSON only."
    )

    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        payload = [
            {
                "generated": item["generated"],
                "ground_truth": item["ground_truth"],
            }
            for item in batch
        ]

        user_prompt = (
            "For each pair, decide if they refer to the same physical facility. "
            "If either side has missing values, use best judgment. "
            "Reply with JSON in the form {\"decisions\":[true,false,...]} in the same order.\n" 
            f"Pairs: {json.dumps(payload, ensure_ascii=True)}"
        )

        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        text = response.choices[0].message.content or ""
        try:
            parsed = json.loads(text)
            batch_decisions = parsed.get("decisions", [])
        except json.JSONDecodeError:
            batch_decisions = []

        if len(batch_decisions) != len(batch):
            batch_decisions = [False] * len(batch)

        decisions.extend(bool(x) for x in batch_decisions)

    return decisions


def evaluate_dataset_matching(
    ground_truth_csv: str | Path,
    generated_csv: str | Path,
    output_dir: str | Path,
    soft_id_cols: Optional[list[str]] = None,
    hard_id_col: Optional[str] = "phone",
    default_region: str = "CA",
    vector_threshold: float = 0.80,
    auto_match_threshold: float = 0.90,
    auto_no_match_threshold: float = 0.60,
    use_llm_judge: bool = True,
) -> dict:
    """
    Evaluate generated dataset against ground truth using matching algorithm.
    
    Args:
        ground_truth_csv: Path to ground truth CSV
        generated_csv: Path to generated dataset CSV
        output_dir: Directory to save output CSVs
        soft_id_cols: Columns to use for soft matching (default: ["address", "name"])
        hard_id_col: Column for hard ID matching (default: "phone")
        default_region: Default region for phone normalization
        vector_threshold: Threshold for vector similarity blocking
        auto_match_threshold: Threshold for automatic match decision
        auto_no_match_threshold: Threshold below which pairs are definitely not matches
        use_llm_judge: Whether to use LLM for gray area decisions
        
    Returns:
        Dictionary with evaluation metrics
    """
    if soft_id_cols is None:
        soft_id_cols = ["address", "name"]
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load datasets
    print("📊 Loading datasets...")
    df_gt = pd.read_csv(ground_truth_csv)
    df_gen = pd.read_csv(generated_csv)
    
    print(f"Ground truth records: {len(df_gt)}")
    print(f"Generated records: {len(df_gen)}")
    
    # Normalize soft columns for both datasets
    for col in soft_id_cols:
        if col not in df_gt.columns:
            df_gt[col] = ""
        if col not in df_gen.columns:
            df_gen[col] = ""
        df_gt[f"__norm_{col}"] = df_gt[col].apply(_normalize_text)
        df_gen[f"__norm_{col}"] = df_gen[col].apply(_normalize_text)
    
    # Normalize hard ID column if provided
    hard_set_col = None
    if hard_id_col and hard_id_col in df_gt.columns and hard_id_col in df_gen.columns:
        hard_set_col = f"__phone_set_{hard_id_col}"
        df_gt[hard_set_col] = df_gt[hard_id_col].apply(lambda v: _normalize_phone_set(v, default_region))
        df_gen[hard_set_col] = df_gen[hard_id_col].apply(lambda v: _normalize_phone_set(v, default_region))
    
    # Create combined text for embedding
    df_gt["__soft_text"] = df_gt[[f"__norm_{c}" for c in soft_id_cols]].agg(" ".join, axis=1).str.strip()
    df_gen["__soft_text"] = df_gen[[f"__norm_{c}" for c in soft_id_cols]].agg(" ".join, axis=1).str.strip()
    
    # Generate embeddings for both datasets
    print("🧮 Generating embeddings...")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    
    all_texts = df_gt["__soft_text"].tolist() + df_gen["__soft_text"].tolist()
    all_embeddings = model.encode(
        all_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    all_embeddings = np.asarray(all_embeddings)
    
    gt_embeddings = all_embeddings[:len(df_gt)]
    gen_embeddings = all_embeddings[len(df_gt):]
    
    # Find candidate matches using vector similarity
    print("🔍 Finding candidate matches...")
    radius = 1.0 - vector_threshold
    nn = NearestNeighbors(metric="cosine", radius=radius)
    nn.fit(gt_embeddings)
    _, indices = nn.radius_neighbors(gen_embeddings, radius=radius)
    
    # Build candidate pairs (gen_idx, gt_idx)
    candidate_pairs: set[tuple[int, int]] = set()
    for gen_idx, gt_neighbors in enumerate(indices):
        for gt_idx in gt_neighbors:
            candidate_pairs.add((gen_idx, gt_idx))

    # Add phone-based candidates to handle multiple numbers in one field
    if hard_set_col:
        phone_to_gt: dict[str, list[int]] = {}
        for gt_idx, phone_set in enumerate(df_gt[hard_set_col]):
            for phone in phone_set:
                phone_to_gt.setdefault(phone, []).append(gt_idx)

        for gen_idx, phone_set in enumerate(df_gen[hard_set_col]):
            for phone in phone_set:
                for gt_idx in phone_to_gt.get(phone, []):
                    candidate_pairs.add((gen_idx, gt_idx))
    
    print(f"Found {len(candidate_pairs)} candidate pairs")
    
    # Score candidate pairs using fuzzy matching
    print("📊 Scoring candidate pairs...")
    confirmed_matches: set[tuple[int, int]] = set()
    gray_pairs: list[tuple[int, int]] = []
    
    for gen_idx, gt_idx in candidate_pairs:
        # Phone overlap is a strong signal of a match
        if hard_set_col:
            gen_phones = df_gen.at[gen_idx, hard_set_col]
            gt_phones = df_gt.at[gt_idx, hard_set_col]
            if gen_phones and gt_phones and gen_phones.intersection(gt_phones):
                confirmed_matches.add((gen_idx, gt_idx))
                continue

        # Skip if either has no soft text
        if not df_gen.at[gen_idx, "__soft_text"] or not df_gt.at[gt_idx, "__soft_text"]:
            gray_pairs.append((gen_idx, gt_idx))
            continue
        
        # Calculate scores for each column
        scores = []
        missing_data = False
        for col in soft_id_cols:
            a = df_gen.at[gen_idx, f"__norm_{col}"]
            b = df_gt.at[gt_idx, f"__norm_{col}"]
            score = _column_score(a, b)
            if score is None:
                missing_data = True
                break
            scores.append(score)
        
        if missing_data or not scores:
            gray_pairs.append((gen_idx, gt_idx))
            continue
        
        # Take minimum score (most conservative)
        final_score = min(scores)
        
        if final_score >= auto_match_threshold:
            confirmed_matches.add((gen_idx, gt_idx))
        elif final_score < auto_no_match_threshold:
            # Definitely not a match
            continue
        else:
            # Gray area - needs LLM judgment
            gray_pairs.append((gen_idx, gt_idx))
    
    print(f"Auto-confirmed matches: {len(confirmed_matches)}")
    print(f"Gray area pairs: {len(gray_pairs)}")
    
    # Use LLM judge for gray area pairs
    if gray_pairs and use_llm_judge:
        print("🤖 Using LLM judge for gray area pairs...")
        pair_payload = []
        for gen_idx, gt_idx in gray_pairs:
            cols_to_include = soft_id_cols + ([hard_id_col] if hard_id_col else [])
            pair_payload.append({
                "generated": df_gen.loc[gen_idx, cols_to_include].to_dict(),
                "ground_truth": df_gt.loc[gt_idx, cols_to_include].to_dict(),
                "index_pair": (gen_idx, gt_idx),
            })
        
        decisions = _llm_judge_match(pair_payload)
        for payload, is_match in zip(pair_payload, decisions):
            if is_match:
                confirmed_matches.add(payload["index_pair"])
    
    # Build match mapping: gen_idx -> gt_idx
    # Each generated record can match at most one ground truth record (take best match)
    gen_to_gt_match: dict[int, int] = {}
    matched_gt_indices: set[int] = set()
    
    for gen_idx, gt_idx in confirmed_matches:
        # If this generated record already has a match, skip
        # (In a more sophisticated version, we could keep the best match)
        if gen_idx not in gen_to_gt_match:
            gen_to_gt_match[gen_idx] = gt_idx
            matched_gt_indices.add(gt_idx)
    
    # Calculate metrics
    print("\n📈 Calculating metrics...")
    
    # True Positives: generated records that matched ground truth
    tp_indices = list(gen_to_gt_match.keys())
    true_positives = len(tp_indices)
    
    # False Positives: generated records that didn't match any ground truth
    fp_indices = [i for i in range(len(df_gen)) if i not in gen_to_gt_match]
    false_positives = len(fp_indices)
    
    # False Negatives: ground truth records that weren't matched by any generated record
    fn_indices = [i for i in range(len(df_gt)) if i not in matched_gt_indices]
    false_negatives = len(fn_indices)
    
    # Calculate precision, recall, F1
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Save True Positives CSV
    df_tp = df_gen.iloc[tp_indices].drop(columns=[c for c in df_gen.columns if c.startswith("__")])
    tp_output_path = output_dir / "true_positives.csv"
    df_tp.to_csv(tp_output_path, index=False)
    
    # Save False Positives CSV
    df_fp = df_gen.iloc[fp_indices].drop(columns=[c for c in df_gen.columns if c.startswith("__")])
    fp_output_path = output_dir / "false_positives.csv"
    df_fp.to_csv(fp_output_path, index=False)
    
    # Save False Negatives CSV (from ground truth)
    df_fn = df_gt.iloc[fn_indices].drop(columns=[c for c in df_gt.columns if c.startswith("__")])
    fn_output_path = output_dir / "false_negatives.csv"
    df_fn.to_csv(fn_output_path, index=False)
    
    # Print results
    print("\n" + "="*60)
    print("📊 EVALUATION RESULTS")
    print("="*60)
    print(f"Ground Truth Records:     {len(df_gt)}")
    print(f"Generated Records:        {len(df_gen)}")
    print("-"*60)
    print(f"True Positives (TP):      {true_positives}")
    print(f"False Positives (FP):     {false_positives}")
    print(f"False Negatives (FN):     {false_negatives}")
    print("-"*60)
    print(f"Precision:                {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall:                   {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1 Score:                 {f1:.4f}")
    print("="*60)
    print("\n💾 Output files saved:")
    print(f"   True Positives:  {tp_output_path}")
    print(f"   False Positives: {fp_output_path}")
    print(f"   False Negatives: {fn_output_path}")
    
    return {
        "ground_truth_count": len(df_gt),
        "generated_count": len(df_gen),
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "output_files": {
            "true_positives": str(tp_output_path),
            "false_positives": str(fp_output_path),
            "false_negatives": str(fn_output_path),
        }
    }

# Dataset Evaluation

This notebook evaluates a generated dataset against ground truth using a matching algorithm based on:
- **Vector embeddings** for initial candidate blocking
- **Fuzzy matching** for scoring similarity
- **Optional LLM judge** for uncertain cases

## Metrics Calculated
- **True Positives (TP)**: Generated records that match ground truth
- **False Positives (FP)**: Generated records with no ground truth match
- **False Negatives (FN)**: Ground truth records not found in generated dataset
- **Precision**: TP / (TP + FP) - How many generated records are correct
- **Recall**: TP / (TP + FN) - How many ground truth records were found
- **F1 Score**: Harmonic mean of precision and recall

## Output Files
- `true_positives.csv` - Generated records that matched ground truth
- `false_positives.csv` - Generated records with no match
- `false_negatives.csv` - Ground truth records that weren't found

## Example Usage

In [4]:
# Evaluate a generated dataset
results = evaluate_dataset_matching(
    ground_truth_csv="combined_disability_18_deduped.csv",
    generated_csv="dataset_employment_4.csv",
    output_dir="eval_csv_employment_4",
    soft_id_cols=["address", "name"],
    hard_id_col="phone",
    vector_threshold=0.80,
    auto_match_threshold=0.90,
    auto_no_match_threshold=0.60,
    use_llm_judge=True,
)

📊 Loading datasets...
Ground truth records: 308
Generated records: 252
🧮 Generating embeddings...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

🔍 Finding candidate matches...
Found 13 candidate pairs
📊 Scoring candidate pairs...
Auto-confirmed matches: 12
Gray area pairs: 1
🤖 Using LLM judge for gray area pairs...

📈 Calculating metrics...

📊 EVALUATION RESULTS
Ground Truth Records:     308
Generated Records:        252
------------------------------------------------------------
True Positives (TP):      10
False Positives (FP):     242
False Negatives (FN):     298
------------------------------------------------------------
Precision:                0.0397 (3.97%)
Recall:                   0.0325 (3.25%)
F1 Score:                 0.0357

💾 Output files saved:
   True Positives:  eval_csv_employment_4/true_positives.csv
   False Positives: eval_csv_employment_4/false_positives.csv
   False Negatives: eval_csv_employment_4/false_negatives.csv


## Parameters Explained

- **ground_truth_csv**: Path to the CSV containing the reference/correct dataset
- **generated_csv**: Path to the CSV containing the dataset to evaluate
- **output_dir**: Directory where output CSVs will be saved
- **soft_id_cols**: List of column names to use for fuzzy matching (e.g., ["address", "name"])
- **hard_id_col**: Column name for exact ID matching (e.g., "phone")
- **vector_threshold**: Minimum cosine similarity (0-1) for candidate pairs (default: 0.80)
- **auto_match_threshold**: Minimum score (0-1) to automatically confirm a match (default: 0.90)
- **auto_no_match_threshold**: Maximum score below which pairs are rejected (default: 0.60)
- **use_llm_judge**: Whether to use LLM for borderline cases between the two thresholds (default: True)

## How It Works

1. **Normalization**: Text and phone numbers are normalized for consistent comparison
2. **Vector Blocking**: Embeddings identify candidate pairs above `vector_threshold`
3. **Fuzzy Scoring**: Each candidate pair gets a similarity score using Jaro-Winkler and token set ratio
4. **Automatic Decisions**: 
   - Score ≥ `auto_match_threshold` → Automatic match
   - Score < `auto_no_match_threshold` → Automatic rejection
5. **LLM Judge**: Borderline cases are sent to LLM for final decision
6. **Metrics Calculation**: TP, FP, FN, Precision, Recall, F1 Score

In [11]:


import requests
searxng_url: str = "http://localhost:8080"
query = 'site:211ontario.ca "GreeneStone Centre for Recovery"'
response = requests.get(
                    f"{searxng_url}/search",
                    params={
                        "q": query,
                        "format": "json",
                        "language": "en",
                        "safesearch": 1,
                        "categories": "general"
                    },
                    timeout=12,
                )
if response.status_code == 200:
    results = response.json().get("results", [])
    for result in results:
        print(f"Title: {result.get('title')}")
        print(f"URL: {result.get('url')}")
        print(f"Content: {result.get('content')}\n")
else:
    print(f"Error: Received status code {response.status_code} from SearXNG")

Title: Greenestone Centre for Recovery - 211 Ontario
URL: https://211ontario.ca/service/71104905/greenestone-centre-for-recovery-greenestone-centre-for-recovery/
Content: Inpatient residential addictions facility. Utilizes a wide variety of programs, techniques and treatment approaches. Medication assistance. Individual, family and couples...

Title: Mental Health / Addictions - Addiction treatment - 211 Ontario
URL: https://211ontario.ca/results/?topicPath=124&latitude=43.256601&longitude=-79.868947&pg=22
Content: Greenestone Centre for Recovery · 1-877-762-5501. Office: 705-710-2230. Office: 1-844-955-5501 · Visit Website. 3571 Muskoka Rd 169. Muskoka Lakes, ON, P0C 1A0 ...

Title: Greenestone Centre for Recovery - Agency Profile
URL: https://211ontario.ca/service/71104902/agency/greenestone-centre-for-recovery/
Content: Greenestone Centre for Recovery Agency Profile Home Agency Description An inpatient residential addictions facility Programs / Services Greenestone Centre for Recove

In [1]:
from pathlib import Path
import json
import os

import pandas as pd
import requests
from openai import OpenAI
from rapidfuzz import fuzz


def _pick_name_column(df: pd.DataFrame, explicit_name_col: str | None = None) -> str:
    if explicit_name_col and explicit_name_col in df.columns:
        return explicit_name_col

    candidates = [
        "Name",
        "name",
        "institution_name",
        "Institution",
        "organization",
        "Organization",
    ]
    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "Could not find a name column. Pass `name_col` explicitly. "
        f"Available columns: {list(df.columns)}"
    )


def _search_searxng_211(
    institution_name: str,
    searxng_url: str = "http://localhost:8080",
    timeout: int = 20,
) -> dict:
    query = f'site:211ontario.ca "{institution_name}"'
    # print(f"🔎 Searching SearXNG for: {query}")
    endpoint = f"{searxng_url.rstrip('/')}/search"

    try:
        response = requests.get(
            endpoint,
            params={"q": query, "format": "json"},
            timeout=timeout,
        )
        response.raise_for_status()
        payload = response.json()
    except Exception:
        return {
            "query": query,
            "hit_count": 0,
            "best_title": "",
            "best_url": "",
            "best_snippet": "",
            "best_score": 0.0,
            "results": [],
        }

    results = payload.get("results", []) or []
    if not results:
        return {
            "query": query,
            "hit_count": 0,
            "best_title": "",
            "best_url": "",
            "best_snippet": "",
            "best_score": 0.0,
            "results": [],
        }

    best = None
    best_score = -1.0

    for item in results:
        title = str(item.get("title", "") or "")
        snippet = str(item.get("content", "") or "")
        target_text = f"{title} {snippet}".strip()
        score = float(fuzz.token_set_ratio(institution_name, target_text))
        if score > best_score:
            best_score = score
            best = item

    return {
        "query": query,
        "hit_count": len(results),
        "best_title": str(best.get("title", "") or "") if best else "",
        "best_url": str(best.get("url", "") or "") if best else "",
        "best_snippet": str(best.get("content", "") or "") if best else "",
        "best_score": max(0.0, best_score),
        "results": results,
    }


def _llm_verify_211_match(
    institution_name: str,
    best_title: str,
    best_snippet: str,
    best_url: str,
    model: str | None = None,
) -> tuple[bool, str]:
    client = OpenAI()
    model = model or os.getenv("OPENAI_MODEL", "gpt-5-nano")

    system_prompt = (
        "You are a strict entity-matching judge for directory verification. "
        "Decide whether the generated institution name refers to the same real-world institution "
        "as the search evidence from 211ontario.ca. Return JSON only."
    )

    user_prompt = (
        "Generated institution name:\n"
        f"{institution_name}\n\n"
        "Search evidence:\n"
        f"Title: {best_title}\n"
        f"URL: {best_url}\n"
        f"Snippet: {best_snippet}\n\n"
        "Decision rules:\n"
        "- true only if they clearly refer to the same organization/service\n"
        "- false if ambiguous, different branch with different org identity, or unrelated\n\n"
        'Return exactly: {"match": true/false, "reason": "short reason"}'
    )

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        text = resp.choices[0].message.content or ""
        parsed = json.loads(text)
        match = bool(parsed.get("match", False))
        reason = str(parsed.get("reason", "llm_no_reason"))
        return match, reason
    except Exception as exc:
        return False, f"llm_error: {exc}"


def verify_false_positives_with_searxng(
    false_positive_csv: str | Path,
    output_dir: str | Path = "eval_csvs",
    name_col: str | None = None,
    searxng_url: str = "http://localhost:8080",
    auto_pass_threshold: float = 90.0,
    llm_min_threshold: float = 60.0,
    llm_max_threshold: float = 89.0,
    llm_model: str | None = None,
    use_llm: bool = True,
) -> dict:
    """
    Verify false-positive records against 211Ontario using SearXNG and threshold tiers.

    Tiers:
      - score >= 90: auto-pass -> verified_211.csv
      - 60 <= score <= 89: LLM judge zone
      - score < 60 or no results: auto-fail -> unverified_211.csv
    """
    false_positive_csv = Path(false_positive_csv)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(false_positive_csv)
    name_col = _pick_name_column(df, name_col)

    verified_rows = []
    unverified_rows = []

    for _, row in df.iterrows():
        record = row.to_dict()
        institution_name = str(record.get(name_col, "") or "").strip()

        if not institution_name:
            record.update(
                {
                    "verification_decision": "unverified",
                    "verification_reason": "missing_name",
                    "search_query": "",
                    "search_hit_count": 0,
                    "best_similarity": 0.0,
                    "best_result_title": "",
                    "best_result_url": "",
                    "best_result_snippet": "",
                }
            )
            unverified_rows.append(record)
            continue

        search = _search_searxng_211(institution_name, searxng_url=searxng_url)
        score = float(search["best_score"])
        hit_count = int(search["hit_count"])

        base_meta = {
            "search_query": search["query"],
            "search_hit_count": hit_count,
            "best_similarity": score,
            "best_result_title": search["best_title"],
            "best_result_url": search["best_url"],
            "best_result_snippet": search["best_snippet"],
        }

        if hit_count == 0:
            record.update({"verification_decision": "unverified", "verification_reason": "no_results", **base_meta})
            unverified_rows.append(record)
            continue

        if score >= auto_pass_threshold:
            record.update({"verification_decision": "verified", "verification_reason": "auto_pass_high_similarity", **base_meta})
            verified_rows.append(record)
            continue

        if score < llm_min_threshold:
            record.update({"verification_decision": "unverified", "verification_reason": "auto_fail_low_similarity", **base_meta})
            unverified_rows.append(record)
            continue

        if llm_min_threshold <= score <= llm_max_threshold and use_llm:
            llm_match, llm_reason = _llm_verify_211_match(
                institution_name=institution_name,
                best_title=search["best_title"],
                best_snippet=search["best_snippet"],
                best_url=search["best_url"],
                model=llm_model,
            )
            if llm_match:
                record.update({"verification_decision": "verified", "verification_reason": f"llm_pass:{llm_reason}", **base_meta})
                verified_rows.append(record)
            else:
                record.update({"verification_decision": "unverified", "verification_reason": f"llm_fail:{llm_reason}", **base_meta})
                unverified_rows.append(record)
            continue

        record.update({"verification_decision": "unverified", "verification_reason": "llm_disabled", **base_meta})
        unverified_rows.append(record)

    verified_df = pd.DataFrame(verified_rows)
    unverified_df = pd.DataFrame(unverified_rows)

    verified_path = output_dir / "verified_211.csv"
    unverified_path = output_dir / "unverified_211.csv"

    verified_df.to_csv(verified_path, index=False)
    unverified_df.to_csv(unverified_path, index=False)

    total = len(df)
    verified_count = len(verified_df)
    unverified_count = len(unverified_df)

    print("\n✅ 211 Verification complete")
    print(f"Total input rows: {total}")
    print(f"Verified: {verified_count}")
    print(f"Unverified: {unverified_count}")
    print(f"Saved: {verified_path}")
    print(f"Saved: {unverified_path}")

    return {
        "input_rows": total,
        "verified_rows": verified_count,
        "unverified_rows": unverified_count,
        "verified_csv": str(verified_path),
        "unverified_csv": str(unverified_path),
    }




In [2]:
summary = verify_false_positives_with_searxng(
    false_positive_csv="dataset_employment_3.csv",
    output_dir="eval_csv_employment_3",
    name_col="name",  
    searxng_url="http://localhost:8080",
    use_llm=True,
)
summary


✅ 211 Verification complete
Total input rows: 255
Verified: 4
Unverified: 251
Saved: eval_csv_employment_3/verified_211.csv
Saved: eval_csv_employment_3/unverified_211.csv


{'input_rows': 255,
 'verified_rows': 4,
 'unverified_rows': 251,
 'verified_csv': 'eval_csv_employment_3/verified_211.csv',
 'unverified_csv': 'eval_csv_employment_3/unverified_211.csv'}

In [ ]:
summary = verify_false_positives_with_searxng(
    false_positive_csv="dataset_learning_disabilities.csv",
    output_dir="eval_csvs",
    name_col="name",  
    searxng_url="http://localhost:8080",
    use_llm=True,
)
summary

In [4]:
from pathlib import Path
import json
import os
from dotenv import find_dotenv, load_dotenv
import pandas as pd
import requests
from rapidfuzz import fuzz

load_dotenv(find_dotenv())

def _pick_any_column(df: pd.DataFrame, candidates: list[str], explicit: str | None = None) -> str | None:
    if explicit:
        return explicit if explicit in df.columns else None
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _build_places_query(name: str, address: str, city: str) -> str:
    pieces = [name, address, city, "Ontario", "Canada"]
    return ", ".join([p for p in pieces if p])


def _google_places_text_search(query: str, api_key: str, timeout: int = 20) -> dict:
    endpoint = "https://maps.googleapis.com/maps/api/place/textsearch/json"
    try:
        resp = requests.get(
            endpoint,
            params={
                "query": query,
                "key": api_key,
                "region": "ca",
            },
            timeout=timeout,
        )
        resp.raise_for_status()
        payload = resp.json()
        return payload if isinstance(payload, dict) else {}
    except Exception as exc:
        return {"status": "REQUEST_ERROR", "error_message": str(exc), "results": []}


def _score_candidate(name: str, address: str, candidate: dict) -> dict:
    g_name = str(candidate.get("name", "") or "")
    g_addr = str(candidate.get("formatted_address", "") or "")
    name_score = float(fuzz.token_set_ratio(name, g_name)) if name and g_name else 0.0
    addr_score = float(fuzz.token_set_ratio(address, g_addr)) if address and g_addr else 0.0
    combined = (0.75 * name_score + 0.25 * addr_score) if address else name_score

    return {
        "name_score": name_score,
        "addr_score": addr_score,
        "combined_score": combined,
    }


def _types_allowed(place_types: list[str], allowed_types: list[str]) -> bool:
    place_set = {str(t).strip().lower() for t in place_types if t}
    allow_set = {str(t).strip().lower() for t in allowed_types if t}

    for place_type in place_set:
        if place_type in allow_set:
            return True
        for allowed in allow_set:
            if allowed in place_type:
                return True
    return False


def verify_with_google_places(
    input_csv: str | Path,
    output_dir: str | Path = "eval_csvs",
    name_col: str | None = None,
    address_col: str | None = None,
    city_col: str | None = None,
    api_key: str | None = None,
    timeout: int = 20,
) -> dict:
    """
    Route unverified/false-positive rows through Google Places and split to verified/unverified CSVs.

    3-step logic:
      1) Physical Existence Check: requires a valid place_id.
      2) Domain Constraint Check: Google `types` must match whitelist.
      3) Operational Status Check: CLOSED_PERMANENTLY/TEMPORARILY -> unverified.

    Outputs:
      - verified_google_places.csv
      - unverified_google_places.csv
    """
    api_key = api_key or os.getenv("GOOGLE_PLACES_API_KEY")
    if not api_key:
        raise ValueError("Missing Google Places key. Set GOOGLE_PLACES_API_KEY or pass api_key=...")

    # Extended whitelist: includes mental health/addiction-related types
    allowed_types = [
        "hospital",
        "health",
        "doctor",
        "pharmacy",
        "local_government_office",
        "social_service",
        "disability_center",
        "rehabilitation_center",
        "disability",
        "physical_therapy",
    ]

    closed_statuses = {"CLOSED_PERMANENTLY", "CLOSED_TEMPORARILY"}
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the input CSV
    input_csv = Path(input_csv)
    df = pd.read_csv(input_csv)
    if df.empty:
        raise ValueError(f"Input CSV is empty: {input_csv}")

    resolved_name_col = _pick_any_column(
        df,
        ["Name", "name", "institution_name", "Institution", "organization", "Organization"],
        name_col,
    )
    if not resolved_name_col:
        raise ValueError(f"Could not resolve name column. Available columns: {list(df.columns)}")

    resolved_address_col = _pick_any_column(
        df,
        ["Address", "address", "street_address", "location", "formatted_address"],
        address_col,
    )
    resolved_city_col = _pick_any_column(
        df,
        ["City", "city", "municipality", "town"],
        city_col,
    )

    verified_rows = []
    unverified_rows = []

    for _, row in df.iterrows():
        record = row.to_dict()
        institution_name = str(record.get(resolved_name_col, "") or "").strip()
        address_text = str(record.get(resolved_address_col, "") or "").strip() if resolved_address_col else ""
        city_text = str(record.get(resolved_city_col, "") or "").strip() if resolved_city_col else ""

        query = _build_places_query(institution_name, address_text, city_text)

        base_meta = {
            "google_query": query,
            "google_status": "",
            "google_error_message": "",
            "google_candidate_count": 0,
            "google_place_id": "",
            "google_place_name": "",
            "google_formatted_address": "",
            "google_types": "[]",
            "google_business_status": "",
            "google_name_similarity": 0.0,
            "google_address_similarity": 0.0,
            "google_combined_similarity": 0.0,
            "google_verification_reason": "",
            "unverified_reason": "",
            "mismatched_types": "",
        }

        if not institution_name:
            record.update({**base_meta, "google_verification_reason": "missing_name"})
            unverified_rows.append(record)
            continue

        payload = _google_places_text_search(query=query, api_key=api_key, timeout=timeout)
        status = str(payload.get("status", "") or "")
        error_message = str(payload.get("error_message", "") or "")
        results = payload.get("results", []) or []

        base_meta["google_status"] = status
        base_meta["google_error_message"] = error_message
        base_meta["google_candidate_count"] = len(results)

        if not results:
            record.update({**base_meta, "google_verification_reason": "no_results", "unverified_reason": "No search results", "mismatched_types": "N/A"})
            unverified_rows.append(record)
            continue

        best_candidate = None
        best_scores = None
        best_combined = -1.0

        for candidate in results:
            scores = _score_candidate(institution_name, address_text, candidate)
            if scores["combined_score"] > best_combined:
                best_combined = scores["combined_score"]
                best_candidate = candidate
                best_scores = scores

        if best_candidate is None or best_scores is None:
            record.update({**base_meta, "google_verification_reason": "no_scored_candidate"})
            unverified_rows.append(record)
            continue

        place_id = str(best_candidate.get("place_id", "") or "")
        place_name = str(best_candidate.get("name", "") or "")
        formatted_address = str(best_candidate.get("formatted_address", "") or "")
        place_types = best_candidate.get("types", []) or []
        business_status = str(best_candidate.get("business_status", "UNKNOWN") or "UNKNOWN")

        base_meta.update(
            {
                "google_place_id": place_id,
                "google_place_name": place_name,
                "google_formatted_address": formatted_address,
                "google_types": json.dumps(place_types, ensure_ascii=True),
                "google_business_status": business_status,
                "google_name_similarity": float(best_scores["name_score"]),
                "google_address_similarity": float(best_scores["addr_score"]),
                "google_combined_similarity": float(best_scores["combined_score"]),
            }
        )

        # Step 1: Physical existence check
        if not place_id:
            record.update({**base_meta, "google_verification_reason": "no_place_id"})
            unverified_rows.append(record)
            continue

        # Step 2: Domain constraint check
        if not _types_allowed(place_types, allowed_types):
            record.update({**base_meta, "google_verification_reason": "type_not_whitelisted", "unverified_reason": "Type mismatch", "mismatched_types": json.dumps(place_types, ensure_ascii=True)})
            unverified_rows.append(record)
            continue

        # Step 3: Operational status check
        if business_status in closed_statuses:
            record.update({**base_meta, "google_verification_reason": f"closed_status:{business_status}"})
            unverified_rows.append(record)
            continue

        record.update({**base_meta, "google_verification_reason": "passed_all_google_checks"})
        verified_rows.append(record)

    verified_df = pd.DataFrame(verified_rows)
    unverified_df = pd.DataFrame(unverified_rows)

    verified_path = output_dir / "verified_google_places.csv"
    unverified_path = output_dir / "unverified_google_places.csv"

    verified_df.to_csv(verified_path, index=False)
    unverified_df.to_csv(unverified_path, index=False)

    summary = {
        "input_rows": int(len(df)),
        "verified_rows": int(len(verified_df)),
        "unverified_rows": int(len(unverified_df)),
        "verified_csv": str(verified_path),
        "unverified_csv": str(unverified_path),
        "name_column": resolved_name_col,
        "address_column": resolved_address_col,
        "city_column": resolved_city_col,
    }

    print("\n✅ Google Places verification complete")
    print(f"Input rows: {summary['input_rows']}")
    print(f"Verified rows: {summary['verified_rows']}")
    print(f"Unverified rows: {summary['unverified_rows']}")
    print(f"Saved: {summary['verified_csv']}")
    print(f"Saved: {summary['unverified_csv']}")

    return summary

In [ ]:
# Example usage: Verify records from unverified_211.csv using Google Places
summary = verify_with_google_places(
    input_csv="dataset_learning_disabilities.csv",
    output_dir="eval_csv_learning_disability",
    name_col="name",  # auto-detected if not specified
    address_col="address",
    city_col="city"  # auto-detected if not specified
)

# Display summary
summary


✅ Google Places verification complete
Input rows: 188
Verified rows: 156
Unverified rows: 32
Saved: eval_csv_physical_disability/verified_google_places.csv
Saved: eval_csv_physical_disability/unverified_google_places.csv


{'input_rows': 188,
 'verified_rows': 156,
 'unverified_rows': 32,
 'verified_csv': 'eval_csv_physical_disability/verified_google_places.csv',
 'unverified_csv': 'eval_csv_physical_disability/unverified_google_places.csv',
 'name_column': 'name',
 'address_column': 'address',
 'city_column': 'city'}

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())


def _extract_text(value):
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, dict):
        direct_text = value.get("text")
        if isinstance(direct_text, str) and direct_text.strip():
            return direct_text.strip()
        for key in ("overview", "description", "summary"):
            nested = _extract_text(value.get(key))
            if nested:
                return nested
    return ""


def _combine_text_parts(*values):
    parts = []
    seen = set()
    for value in values:
        text = _extract_text(value)
        if text and text not in seen:
            seen.add(text)
            parts.append(text)
    return "\n\n".join(parts)


def _resolve_required_column(df: pd.DataFrame, target_name: str) -> str:
    lookup = {str(col).strip().lower(): col for col in df.columns}
    resolved = lookup.get(target_name.lower())
    if not resolved:
        raise ValueError(f"Missing required column '{target_name}'. Available: {list(df.columns)}")
    return resolved


def query_google_places_new(query_string, api_key=None, timeout=20):
    """
    Query Google Places (New), enrich the top search hit with Place Details,
    and select the best available text source for downstream keyword matching.

    Fallback order for text selection:
      1. editorialSummary
      2. generativeSummary
      3. reviews + reviewSummary
    """
    if not api_key:
        api_key = os.environ.get("GOOGLE_PLACES_API_KEY")
        if not api_key:
            raise ValueError("API Key not found. Please provide it or set GOOGLE_PLACES_API_KEY.")

    search_url = "https://places.googleapis.com/v1/places:searchText"
    search_headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.id,places.name,places.displayName,places.formattedAddress",
    }
    search_payload = {
        "textQuery": query_string,
        "languageCode": "en",
        "pageSize": 1,
        "regionCode": "CA",
    }

    try:
        search_response = requests.post(
            search_url,
            json=search_payload,
            headers=search_headers,
            timeout=timeout,
        )
        search_response.raise_for_status()
        result = search_response.json()

        places = result.get("places", []) or []
        if not places:
            result["selectedTextSource"] = None
            result["selectedText"] = ""
            return result

        first_match = dict(places[0])
        place_resource_name = first_match.get("name")

        if place_resource_name:
            details_response = requests.get(
                f"https://places.googleapis.com/v1/{place_resource_name}",
                headers={
                    "Content-Type": "application/json",
                    "X-Goog-Api-Key": api_key,
                    "X-Goog-FieldMask": (
                        "id,name,displayName,formattedAddress,types,businessStatus,websiteUri,"
                        "editorialSummary,generativeSummary,reviews,reviewSummary"
                    ),
                },
                timeout=timeout,
            )
            details_response.raise_for_status()
            details_payload = details_response.json()
            first_match.update(details_payload)
            result["places"][0] = first_match

        editorial_text = _combine_text_parts(first_match.get("editorialSummary"))

        generative_summary = first_match.get("generativeSummary") or {}
        generative_text = _combine_text_parts(
            generative_summary.get("overview"),
            generative_summary.get("description"),
            generative_summary,
        )

        review_summary_text = _combine_text_parts(first_match.get("reviewSummary"))
        reviews = first_match.get("reviews", []) or []
        review_texts = [
            _extract_text(review.get("text"))
            for review in reviews[:5]
            if isinstance(review, dict)
        ]
        review_text_block = "\n\n".join([text for text in [review_summary_text, *review_texts] if text])

        if editorial_text:
            selected_text_source = "editorialSummary"
            selected_text = editorial_text
        elif generative_text:
            selected_text_source = "generativeSummary"
            selected_text = generative_text
        elif review_text_block:
            selected_text_source = "reviewsAndReviewSummary"
            selected_text = review_text_block
        else:
            selected_text_source = None
            selected_text = ""

        result["selectedTextSource"] = selected_text_source
        result["selectedText"] = selected_text

        return result

    except requests.exceptions.RequestException as exc:
        print(f"API Request failed: {exc}")
        if hasattr(exc, "response") and exc.response is not None:
            print(f"Error Details: {exc.response.text}")
        return None


def verify_institutions_from_csv(
    input_csv,
    output_dir="eval_csvs",
    keywords=None,
    api_key=None,
    timeout=20,
):
    """
    Validate institutions from CSV using Google Places text extraction + keyword checks.

    Required input columns:
      - name
      - address
      - city
      - phone
      - source_url

    Valid if a keyword appears in either:
      - the selected description text (editorial -> generative -> reviews fallback), or
      - the institution name column.

    Output CSVs:
      - verified_institutions.csv
      - unverified_institutions.csv

    Output columns for both files:
      - name
      - address
      - city
      - phone
      - source_url
      - description
      - reason
    """
    if not api_key:
        api_key = os.environ.get("GOOGLE_PLACES_API_KEY")
        if not api_key:
            raise ValueError("API Key not found. Please provide it or set GOOGLE_PLACES_API_KEY.")

    if keywords is None:
        keywords = [
            "addiction",
            "recovery",
            "rehab",
            "detox",
            "treatment",
            "support",
            "therapy",
            "counselling",
            "counseling",
            "mental health",
        ]

    normalized_keywords = [str(keyword).strip() for keyword in keywords if str(keyword).strip()]

    input_csv = Path(input_csv)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(input_csv)
    if df.empty:
        raise ValueError(f"Input CSV is empty: {input_csv}")

    name_col = _resolve_required_column(df, "name")
    address_col = _resolve_required_column(df, "address")
    city_col = _resolve_required_column(df, "city")
    phone_col = _resolve_required_column(df, "phone")
    source_url_col = _resolve_required_column(df, "source_url")

    query_technique_col = None
    column_lookup = {str(col).strip().lower(): col for col in df.columns}
    if "query_technique" in column_lookup:
        query_technique_col = column_lookup["query_technique"]

    output_columns = [
        "name",
        "address",
        "city",
        "phone",
        "source_url",
        "description",
        "reason",
    ]

    verified_rows = []
    unverified_rows = []
    query_technique_counts: dict[str, dict[str, int]] = {}

    def _increment_query_technique_count(query_technique: str, is_valid: bool) -> None:
        normalized_technique = str(query_technique).strip() if str(query_technique).strip() else "unknown"
        if normalized_technique not in query_technique_counts:
            query_technique_counts[normalized_technique] = {"valid": 0, "invalid": 0}
        key = "valid" if is_valid else "invalid"
        query_technique_counts[normalized_technique][key] += 1

    for _, row in df.iterrows():
        name = "" if pd.isna(row.get(name_col)) else str(row.get(name_col)).strip()
        address = "" if pd.isna(row.get(address_col)) else str(row.get(address_col)).strip()
        city = "" if pd.isna(row.get(city_col)) else str(row.get(city_col)).strip()
        phone = "" if pd.isna(row.get(phone_col)) else str(row.get(phone_col)).strip()
        source_url = "" if pd.isna(row.get(source_url_col)) else str(row.get(source_url_col)).strip()
        query_technique = "unknown"
        if query_technique_col and not pd.isna(row.get(query_technique_col)):
            candidate_technique = str(row.get(query_technique_col)).strip()
            if candidate_technique:
                query_technique = candidate_technique

        query = ", ".join([piece for piece in [name, address, city] if piece])

        base_record = {
            "name": name,
            "address": address,
            "city": city,
            "phone": phone,
            "source_url": source_url,
            "description": "",
            "reason": "",
        }

        if not query:
            base_record["reason"] = "invalid:empty_query"
            unverified_rows.append(base_record)
            _increment_query_technique_count(query_technique, False)
            continue

        payload = query_google_places_new(query, api_key=api_key, timeout=timeout)
        if payload is None:
            base_record["reason"] = "invalid:google_api_request_failed"
            unverified_rows.append(base_record)
            _increment_query_technique_count(query_technique, False)
            continue

        places = payload.get("places", []) or []
        if not places:
            base_record["reason"] = "invalid:no_google_places_match"
            unverified_rows.append(base_record)
            _increment_query_technique_count(query_technique, False)
            continue

        description = str(payload.get("selectedText") or "").strip()
        selected_source = payload.get("selectedTextSource") or "none"
        base_record["description"] = description

        name_lower = name.lower()
        matched_in_name = [
            keyword
            for keyword in normalized_keywords
            if keyword.lower() in name_lower
        ]

        matched_in_description = []
        if description:
            description_lower = description.lower()
            matched_in_description = [
                keyword
                for keyword in normalized_keywords
                if keyword.lower() in description_lower
            ]

        matched_keywords = []
        for keyword in [*matched_in_description, *matched_in_name]:
            if keyword not in matched_keywords:
                matched_keywords.append(keyword)

        if matched_keywords:
            source_parts = []
            if matched_in_description:
                source_parts.append(f"description:{selected_source}")
            if matched_in_name:
                source_parts.append("name")
            source_text = "+".join(source_parts)
            keyword_text = ", ".join(matched_keywords)
            base_record["reason"] = f"valid:keyword_match(source={source_text};keywords={keyword_text})"
            verified_rows.append(base_record)
            _increment_query_technique_count(query_technique, True)
        else:
            if not description:
                base_record["reason"] = "invalid:no_text_available_and_no_keyword_match(source=name)"
            else:
                base_record["reason"] = f"invalid:no_keyword_match(source=description:{selected_source}+name)"
            unverified_rows.append(base_record)
            _increment_query_technique_count(query_technique, False)

    verified_df = pd.DataFrame(verified_rows, columns=output_columns)
    unverified_df = pd.DataFrame(unverified_rows, columns=output_columns)

    verified_path = output_dir / "verified_institutions.csv"
    unverified_path = output_dir / "unverified_institutions.csv"

    verified_df.to_csv(verified_path, index=False)
    unverified_df.to_csv(unverified_path, index=False)

    summary = {
        "input_rows": int(len(df)),
        "verified_rows": int(len(verified_df)),
        "unverified_rows": int(len(unverified_df)),
        "verified_csv": str(verified_path),
        "unverified_csv": str(unverified_path),
    }

    print("\nCSV institution verification complete")
    print(f"Input rows: {summary['input_rows']}")
    print(f"Verified rows: {summary['verified_rows']}")
    print(f"Unverified rows: {summary['unverified_rows']}")
    print(f"Saved: {summary['verified_csv']}")
    print(f"Saved: {summary['unverified_csv']}")

    query_technique_summary_rows = []
    for query_technique, counts in sorted(query_technique_counts.items()):
        valid_count = int(counts.get("valid", 0))
        invalid_count = int(counts.get("invalid", 0))
        query_technique_summary_rows.append({
            "query_technique": query_technique,
            "valid": valid_count,
            "invalid": invalid_count,
            "total": valid_count + invalid_count,
        })

    print("\nQuery technique summary (valid vs invalid):")
    if query_technique_summary_rows:
        query_technique_summary_df = pd.DataFrame(query_technique_summary_rows)
        print(query_technique_summary_df.to_string(index=False))
    else:
        print("No query-technique stats available.")

    summary["query_technique_summary"] = query_technique_summary_rows

    return summary

In [2]:
# Example: verify institutions from CSV using fallback description logic + keyword check
summary = verify_institutions_from_csv(
    input_csv="combined_disability_18_deduped.csv",
    output_dir="combined_disability_18_deduped/instructions",
    keywords=[
        "employment disability",
        "disability employment",
        "employment support",
        "employment services",
        "job coaching",
        "supported employment",
        "vocational rehabilitation",
        "vocational services",
        "job readiness",
        "employment readiness",
        "job placement",
        "workplace accommodation",
        "work accommodations",
        "career counseling",
        "career counselling",
        "skills training",
        "resume support",
        "interview preparation",
        "return to work",
        "inclusive hiring",
        "employment integration",
        "workforce development",
        "assistive technology at work",
        "disability career services",
        "employment resource centre",
        "disability"
        "employment",
        "job",
        "career",
        "work"
    ],
)

summary


CSV institution verification complete
Input rows: 308
Verified rows: 184
Unverified rows: 124
Saved: combined_disability_18_deduped/instructions/verified_institutions.csv
Saved: combined_disability_18_deduped/instructions/unverified_institutions.csv

Query technique summary (valid vs invalid):
query_technique  valid  invalid  total
        unknown    184      124    308


{'input_rows': 308,
 'verified_rows': 184,
 'unverified_rows': 124,
 'verified_csv': 'combined_disability_18_deduped/instructions/verified_institutions.csv',
 'unverified_csv': 'combined_disability_18_deduped/instructions/unverified_institutions.csv',
 'query_technique_summary': [{'query_technique': 'unknown',
   'valid': 184,
   'invalid': 124,
   'total': 308}]}

In [2]:
# Example: verify institutions from CSV using fallback description logic + keyword check
summary = verify_institutions_from_csv(
    input_csv="dataset_employment_3.csv",
    output_dir="eval_csv_employment_3/instructions",
    keywords=[
        "employment disability",
        "disability employment",
        "employment support",
        "employment services",
        "job coaching",
        "supported employment",
        "vocational rehabilitation",
        "vocational services",
        "job readiness",
        "employment readiness",
        "job placement",
        "workplace accommodation",
        "work accommodations",
        "career counseling",
        "career counselling",
        "skills training",
        "resume support",
        "interview preparation",
        "return to work",
        "inclusive hiring",
        "employment integration",
        "workforce development",
        "assistive technology at work",
        "disability career services",
        "employment resource centre",
        "disability"
        "employment",
        "job",
        "career",
        "work"
    ],
)

summary

API Request failed: 503 Server Error: Service Unavailable for url: https://places.googleapis.com/v1/places:searchText
Error Details: {
  "error": {
    "code": 503,
    "message": "The service is currently unavailable.",
    "status": "UNAVAILABLE"
  }
}


CSV institution verification complete
Input rows: 255
Verified rows: 191
Unverified rows: 64
Saved: eval_csv_employment_3/instructions/verified_institutions.csv
Saved: eval_csv_employment_3/instructions/unverified_institutions.csv

Query technique summary (valid vs invalid):
query_technique  valid  invalid  total
artifact_hunter     26       16     42
      broad_net    137       42    179
      deep_dive     28        6     34


{'input_rows': 255,
 'verified_rows': 191,
 'unverified_rows': 64,
 'verified_csv': 'eval_csv_employment_3/instructions/verified_institutions.csv',
 'unverified_csv': 'eval_csv_employment_3/instructions/unverified_institutions.csv',
 'query_technique_summary': [{'query_technique': 'artifact_hunter',
   'valid': 26,
   'invalid': 16,
   'total': 42},
  {'query_technique': 'broad_net', 'valid': 137, 'invalid': 42, 'total': 179},
  {'query_technique': 'deep_dive', 'valid': 28, 'invalid': 6, 'total': 34}]}